# MedLingua — Fine-Tune Gemma 4 for Medical Triage

This notebook fine-tunes **Gemma 4 E4B** on WHO IMCI triage protocols using **Unsloth** (4-bit QLoRA).

**Runtime required:** GPU (T4 free tier works, A100 recommended)

Pipeline:
1. Install dependencies
2. Upload training data
3. Prepare dataset
4. Fine-tune with Unsloth
5. Export model for on-device deployment
6. Download the exported model

## 0. Verify GPU

In [ ]:
!nvidia-smi

## 1. Install Dependencies

In [ ]:
%%capture
# Install Unsloth (handles torch, transformers, peft, trl, etc.)
!pip install unsloth
# Fixes for Colab
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install datasets huggingface_hub sentencepiece protobuf

## 2. Upload Training Data

Upload `training/data/imci_triage_dataset.jsonl` from your project.

**Option A:** Run the cell below to upload via the file picker.

**Option B:** If you cloned the repo, skip this and set the path directly.

In [ ]:
import os
os.makedirs("training/data", exist_ok=True)
os.makedirs("training/output", exist_ok=True)

# Option A: Upload from your machine
try:
    from google.colab import files
    print("Upload imci_triage_dataset.jsonl:")
    uploaded = files.upload()
    for fn in uploaded:
        os.rename(fn, "training/data/imci_triage_dataset.jsonl")
        print(f"Saved: training/data/imci_triage_dataset.jsonl")
except ImportError:
    # Kaggle or already have the file
    print("Not on Colab. Make sure training/data/imci_triage_dataset.jsonl exists.")

# Verify
assert os.path.exists("training/data/imci_triage_dataset.jsonl"), "Dataset file not found!"
with open("training/data/imci_triage_dataset.jsonl") as f:
    lines = f.readlines()
print(f"Dataset loaded: {len(lines)} samples")

## 3. Prepare Dataset

Convert the IMCI data into the chat template format for Gemma fine-tuning.

In [ ]:
import json
import random

SYSTEM_PROMPT = """You are MedLingua, a medical triage assistant for Community Health Workers.
Follow WHO IMCI (Integrated Management of Childhood Illness) protocols.

You MUST use the classify_triage function to provide structured output.

IMPORTANT: You are a triage SUPPORT tool, not a doctor. Always recommend
professional medical consultation for serious conditions."""

def format_chat_template(instruction, input_text, output_text):
    user_content = instruction
    if input_text:
        user_content += f"\n\n{input_text}"
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": output_text},
        ]
    }

# Load and convert
samples = []
with open("training/data/imci_triage_dataset.jsonl") as f:
    for line in f:
        raw = json.loads(line.strip())
        samples.append(format_chat_template(
            raw["instruction"], raw.get("input", ""), raw["output"]
        ))

random.seed(42)
random.shuffle(samples)

# Write output
output_path = "training/data/medlingua_train.jsonl"
with open(output_path, "w") as f:
    for s in samples:
        f.write(json.dumps(s) + "\n")

# Stats
severity_counts = {}
for s in samples:
    parsed = json.loads(s["messages"][-1]["content"])
    sev = parsed["severity"]
    severity_counts[sev] = severity_counts.get(sev, 0) + 1

print(f"Total training samples: {len(samples)}")
print(f"Output: {output_path}")
print("\nSeverity distribution:")
for sev, count in sorted(severity_counts.items()):
    print(f"  {sev:12s}: {count:3d} ({count/len(samples)*100:.1f}%)")

## 4. Fine-Tune with Unsloth

4-bit QLoRA fine-tuning of Gemma 4 E4B. Takes ~20 min on T4, ~10 min on A100.

In [ ]:
from unsloth import FastModel
from datasets import load_dataset

# Load base model
model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/gemma-4-E4B-it-unsloth-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
)
print("Base model loaded!")

In [ ]:
# Apply LoRA adapters
model = FastModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)")

In [ ]:
# Load dataset
dataset = load_dataset("json", data_files="training/data/medlingua_train.jsonl", split="train")
print(f"Training samples: {len(dataset)}")

# Apply chat template
def format_sample(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

dataset = dataset.map(format_sample, remove_columns=dataset.column_names)

# Preview one sample
print("\n--- Sample preview (first 500 chars) ---")
print(dataset[0]["text"][:500])

In [ ]:
from trl import SFTTrainer, SFTConfig
import time

OUTPUT_DIR = "training/output/medlingua-lora"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=False,
    bf16=True,
    max_seq_length=2048,
    logging_steps=5,
    save_steps=50,
    save_total_limit=2,
    seed=42,
    report_to="none",
    dataset_text_field="text",
    packing=True,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=training_args,
)

print("Starting training...")
start = time.time()
trainer.train()
elapsed = time.time() - start
print(f"\nTraining complete! Time: {elapsed/60:.1f} minutes")

In [ ]:
# Save LoRA adapter
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Save training config
config = {
    "base_model": "unsloth/gemma-4-E4B-it-unsloth-bnb-4bit",
    "epochs": 3,
    "lora_rank": 32,
    "lora_alpha": 64,
    "training_samples": len(dataset),
    "training_time_minutes": round(elapsed / 60, 1),
}
with open(f"{OUTPUT_DIR}/medlingua_training_config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"LoRA adapter saved to {OUTPUT_DIR}")

## 5. Test the Fine-Tuned Model

Quick inference test before exporting.

In [ ]:
# Test inference
test_prompt = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": (
        "A 3-year-old male has had diarrhea for 2 days with blood in stool. "
        "He has sunken eyes and is drinking eagerly. Fever of 38.5°C. "
        "Assess and classify using IMCI guidelines.\n\n"
        "Age: 3 years\nGender: Male\n"
        "Symptoms: bloody diarrhea 2 days, sunken eyes, drinks eagerly, fever 38.5°C"
    )},
]

inputs = tokenizer.apply_chat_template(
    test_prompt, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to(model.device)

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    temperature=0.3,
    top_k=20,
    do_sample=True,
)

response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
print("Model response:")
print(response)

# Try to parse as JSON
try:
    parsed = json.loads(response)
    print("\n--- Parsed output ---")
    print(f"Severity:       {parsed['severity']}")
    print(f"Diagnosis:      {parsed['diagnosis']}")
    print(f"Confidence:     {parsed['confidence']}")
    print(f"Recommendation: {parsed['recommendation'][:100]}...")
except json.JSONDecodeError:
    print("\n(Response is not pure JSON — the app's parser handles this)")

## 6. Export Model

Merge LoRA, quantize to int4, and export as GGUF for on-device deployment.

In [ ]:
EXPORT_DIR = "training/output/medlingua-exported"
os.makedirs(EXPORT_DIR, exist_ok=True)

# Export as GGUF (q4_k_m quantization)
print("Exporting as GGUF (int4 quantized)...")
model.save_pretrained_gguf(
    EXPORT_DIR,
    tokenizer,
    quantization_method="q4_k_m",
)

# Find and report the GGUF file
from pathlib import Path
gguf_files = list(Path(EXPORT_DIR).glob("*.gguf"))
for f in gguf_files:
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"Exported: {f.name} ({size_mb:.0f} MB)")

print("\nExport complete!")

In [ ]:
# Also save LoRA adapter separately (smaller, for use with base model)
import shutil

lora_export = os.path.join(EXPORT_DIR, "lora")
os.makedirs(lora_export, exist_ok=True)

for fn in ["adapter_config.json", "adapter_model.safetensors", 
           "adapter_model.bin", "medlingua_training_config.json"]:
    src = os.path.join(OUTPUT_DIR, fn)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(lora_export, fn))
        print(f"Copied: {fn}")

print(f"\nLoRA adapter saved to {lora_export}")

## 7. Download the Model

Download the exported model to your local machine, then push to your Android device.

In [ ]:
# List all exported files
print("Exported files:")
for root, dirs, filenames in os.walk(EXPORT_DIR):
    for fn in filenames:
        fp = os.path.join(root, fn)
        size = os.path.getsize(fp) / (1024 * 1024)
        print(f"  {os.path.relpath(fp, EXPORT_DIR):50s} {size:8.1f} MB")

In [ ]:
# Download GGUF model file (Colab only)
try:
    from google.colab import files as colab_files
    if gguf_files:
        print(f"Downloading {gguf_files[0].name}...")
        colab_files.download(str(gguf_files[0]))
    else:
        print("No GGUF file found. Check export step above.")
except ImportError:
    print("Not on Colab. Download files manually from the file browser.")
    print(f"Files are in: {EXPORT_DIR}")

## 8. (Optional) Push to HuggingFace Hub

In [ ]:
# Uncomment and set your repo name to push to HuggingFace Hub

# HUB_REPO = "your-username/medlingua-gemma-lora"  # <-- Change this

# from huggingface_hub import login
# login()  # Enter your HF token

# model.push_to_hub_merged(HUB_REPO, tokenizer, save_method="merged_16bit")
# model.push_to_hub_gguf(HUB_REPO, tokenizer, quantization_method="q4_k_m")
# print(f"Pushed to https://huggingface.co/{HUB_REPO}")

---

## Next Steps

After downloading the GGUF model file:

```bash
# Push to your Android device
adb push <model_file>.gguf /storage/emulated/0/Documents/models/

# Or use the project script
./scripts/download_model.sh --push
```

Launch MedLingua — the app will auto-detect the model and switch from demo mode to real inference.